# Reanalysis of published dopaminergic vesicle proteomes



| Panel | Dataset | Comparison | Statistic reported by authors |
|---|---|---|---|
| a | Hobson et al. 2022, eLife (Fig. 2—source data 5) | APEX2+ striatum vs bulk striatum | log2FC, BH-adjusted q |
| b | Paget-Blanc et al. 2022, Nat Commun (Supp. Table 3) | DA-FASS synaptosomes vs bulk synaptosomes | abundance ratio, BH-adjusted p |
| c | Asmerian et al. 2026, Sci Adv (Tables S1, S2) | VGLUT2+ vs VMAT2+ vesicles, striatal LP2 and P4 | mean log2 ratio, uncorrected two-tailed p |

No statistics are recomputed here. Effect sizes and significance values are
taken as published.



## 1. Setup


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

#ADD DATA PATH


DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))
OUT_DIR = Path("figures")
OUT_DIR.mkdir(exist_ok=True)

FILES = {
    "hobson":     DATA_DIR / "hobson_elife-70921-fig2-data5-v2.xlsx",
    "pagetblanc": DATA_DIR / "pagetblanc_41467_2022_30776_MOESM1_TableS3.xlsx",
    "asmerian_lp2": DATA_DIR / "asmerian_adz6836_table_s1.xlsx",   # striatal LP2
    "asmerian_p4":  DATA_DIR / "asmerian_adz6836_table_s2.xlsx",   # striatal P4
}

print("pandas", pd.__version__, "| numpy", np.__version__, "| matplotlib", mpl.__version__)


## 2. Protein panel

The three datasets use mouse gene symbols with inconsistent capitalisation, so
everything is matched on the uppercased symbol. `PANEL` maps that symbol to the
protein name used in the figure.


In [ ]:
PANEL = {
    # dopaminergic identity markers
    "SLC18A2": "VMAT2",
    "TH":      "TH",
    "DDC":     "DDC",
    "SLC6A3":  "DAT",
    "SLC10A4": "Slc10a4",
    "DBH":     "DBH",
    # SV2 isoforms
    "SV2A": "SV2A",
    "SV2B": "SV2B",
    "SV2C": "SV2C",
    # synaptophysin / synaptogyrin family
    "SYP":    "Synaptophysin",
    "SYNGR1": "Syngr1",
    "SYNGR2": "Syngr2",
    "SYNGR3": "Syngr3",
    # PD-associated and SV proteins
    "SYNJ1":  "Synj1",
    "DNAJC6": "Dnajc6",
    "SNCA":   "aSyn",
    "SNAP25": "Snap25",
    # glutamatergic marker — orientation for panel c
    "SLC17A6": "VGLUT2",
}

def to_symbol(x):
    """Uppercase a gene symbol so the three datasets can be matched."""
    return str(x).strip().upper()

def take_panel(df, symbol_col):
    """Keep only panel proteins and attach the figure label."""
    out = df.copy()
    out["symbol"] = out[symbol_col].map(to_symbol)
    out = out[out["symbol"].isin(PANEL)]
    out["protein"] = out["symbol"].map(PANEL)
    return out

def report(name, df):
    missing = sorted(set(PANEL) - set(df["symbol"]))
    print(f"{name}: {len(df)}/{len(PANEL)} panel proteins found")
    if missing:
        print("   not detected:", ", ".join(missing))


## 3. Load the three datasets

Each block reads one file and reduces it to three columns: `protein`, the
effect size, and the significance value. Nothing else is carried forward.


In [ ]:
# --- Panel a: Hobson 2022 -----------------------------------------------
# Sheet "Bulk_v_APEX_Str" is the striatal comparison. Positive log2FC means
# enriched in the APEX2-labelled dopaminergic compartment.
hobson = pd.read_excel(FILES["hobson"], sheet_name="Bulk_v_APEX_Str")
hobson = take_panel(hobson, "Genes")
hobson = hobson[["protein", "symbol", "Log2FC", "qvalue"]]
hobson.columns = ["protein", "symbol", "effect", "pvalue"]

report("Hobson", hobson)
hobson.sort_values("effect", ascending=False)


In [ ]:
# --- Panel b: Paget-Blanc 2022 ------------------------------------------
# The first row of the sheet is a title, so the real header is row 2.
# Authors report a linear abundance ratio; log2 it to match panel a.
pb = pd.read_excel(FILES["pagetblanc"], sheet_name="Table 1", header=1)
pb = take_panel(pb, "Gene Symbol")

RATIO = "Abundance Ratio: (DA-FASS) / (SYN)"
ADJP = "Abundance Ratio Adj. P-Value: (DA-FASS) / (SYN)"

pagetblanc = pd.DataFrame({
    "protein": pb["protein"],
    "symbol": pb["symbol"],
    "effect": np.log2(pb[RATIO].astype(float)),
    "pvalue": pb[ADJP].astype(float),
})

report("Paget-Blanc", pagetblanc)
pagetblanc.sort_values("effect", ascending=False)


In [ ]:
# --- Panel c: Asmerian 2026 ---------------------------------------------
# Both tables have six rows of preamble. The "Average" column is the mean
# log2(VGLUT2/VMAT2) ratio, so NEGATIVE = enriched on VMAT2 vesicles.
# Significance is given as -log10(p); convert back to p.
def read_asmerian(path):
    df = pd.read_excel(path, header=6)
    df = take_panel(df, "Gene")
    out = pd.DataFrame({
        "protein": df["protein"],
        "symbol": df["symbol"],
        "effect": df["Average"].astype(float),
        "pvalue": 10 ** (-df["-LOG10(P)"].astype(float)),
    })
    return out

asm_lp2 = read_asmerian(FILES["asmerian_lp2"])
asm_p4 = read_asmerian(FILES["asmerian_p4"])

report("Asmerian LP2", asm_lp2)
report("Asmerian P4", asm_p4)



In [ ]:
# Asmerian: merge the two fractions side by side, LP2 first.
asmerian = pd.merge(
    asm_lp2.rename(columns={"effect": "LP2_effect", "pvalue": "LP2_pvalue"}),
    asm_p4.rename(columns={"effect": "P4_effect", "pvalue": "P4_pvalue"}),
    on=["protein", "symbol"], how="outer",
)

# Synj1 and Dnajc6 are detected in a single replicate, so no p-value can be
# computed for them. They are excluded from the figure as stated in
# the Methods.
SINGLE_REPLICATE = ["Synj1", "Dnajc6"]
asmerian = asmerian[~asmerian["protein"].isin(SINGLE_REPLICATE)]
asmerian = asmerian.dropna(subset=["LP2_effect"])

asmerian.sort_values("LP2_effect")


## 4. Source data table

Table containing values that appears in the figure.


In [ ]:
source_data = pd.concat([
    hobson.assign(dataset="Hobson 2022", comparison="APEX2+ striatum / bulk striatum",
                  statistic="BH-adjusted q"),
    pagetblanc.assign(dataset="Paget-Blanc 2022", comparison="DA-FASS / bulk synaptosomes",
                      statistic="BH-adjusted p"),
    asm_lp2.assign(dataset="Asmerian 2026", comparison="VGLUT2+ / VMAT2+ vesicles, LP2",
                   statistic="uncorrected two-tailed p"),
    asm_p4.assign(dataset="Asmerian 2026", comparison="VGLUT2+ / VMAT2+ vesicles, P4",
                  statistic="uncorrected two-tailed p"),
], ignore_index=True)

source_data = source_data[["dataset", "comparison", "protein", "symbol",
                           "effect", "pvalue", "statistic"]]
source_data.to_csv(OUT_DIR / "source_data_figure5.csv", index=False)

print(f"wrote {len(source_data)} rows")
source_data.head()


## 5. Figure

Each panel is drawn separately and saved separetly, then assembled in
Illustrator.



In [ ]:
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

BLUE = "#2166ac"
RED = "#b2182b"

def significance_mark(p):
    """Asterisk tiers as used throughout the manuscript."""
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

def heat_panel(proteins, effects, pvalues, column_labels, title, cbar_label,
               filename, arrows=None, figsize=(2.4, 5.2)):
    """Draw one heatmap column (or pair of columns) and save it as PDF + PNG.

    proteins      row labels, top to bottom
    effects       (n_proteins, n_columns) array of effect sizes
    pvalues       matching array of significance values
    arrows        optional [(y_start, y_end, colour, text), ...] in axes
                  fraction, drawn to the right of the panel
    """
    effects = np.asarray(effects, dtype=float)
    pvalues = np.asarray(pvalues, dtype=float)
    limit = np.nanmax(np.abs(effects))

    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    image = ax.imshow(effects, aspect="auto", cmap="RdBu_r",
                      vmin=-limit, vmax=limit)

    for row in range(effects.shape[0]):
        for col in range(effects.shape[1]):
            value = effects[row, col]
            if np.isnan(value):
                continue
            mark = significance_mark(pvalues[row, col])
            if not mark:
                continue
            # dark cells need light text
            colour = "white" if abs(value) > limit * 0.55 else "black"
            ax.text(col, row, mark, ha="center", va="center",
                    fontsize=6.5, color=colour,
                    style="italic" if mark == "ns" else "normal")

    ax.set_xticks(range(effects.shape[1]))
    ax.set_xticklabels(column_labels, fontsize=7)
    ax.set_yticks(range(len(proteins)))
    ax.set_yticklabels(proteins, fontsize=7.5)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(title, fontsize=8, pad=14)

    bar = fig.colorbar(image, ax=ax, orientation="horizontal",
                       location="bottom", fraction=0.06, pad=0.08)
    bar.set_label(cbar_label, fontsize=6.2)
    bar.ax.tick_params(labelsize=5.8)

    for y_start, y_end, colour, text in (arrows or []):
        ax.annotate("", xy=(1.16, y_end), xytext=(1.16, y_start),
                    xycoords="axes fraction",
                    arrowprops=dict(arrowstyle="-|>", color=colour, lw=2.0,
                                    mutation_scale=18))
        ax.text(1.23, (y_start + y_end) / 2, text, transform=ax.transAxes,
                rotation=90, fontsize=8, color=colour, ha="center", va="center")

    fig.savefig(OUT_DIR / f"{filename}.pdf", bbox_inches="tight")
    fig.savefig(OUT_DIR / f"{filename}.png", dpi=600, bbox_inches="tight")
    plt.show()


In [ ]:
# Panel a — most enriched at the top.
d = hobson.dropna(subset=["effect"]).sort_values("effect", ascending=False)

heat_panel(
    proteins=d["protein"].tolist(),
    effects=d[["effect"]].values,
    pvalues=d[["pvalue"]].values,
    column_labels=[""],
    title="Dopaminergic terminal enrichment\n(APEX2+ vs bulk striatum)",
    cbar_label="Log2FC (APEX2+ / bulk)",
    filename="panel_a_hobson",
)


In [ ]:
# Panel b — same ordering convention as panel a.
d = pagetblanc.dropna(subset=["effect"]).sort_values("effect", ascending=False)

heat_panel(
    proteins=d["protein"].tolist(),
    effects=d[["effect"]].values,
    pvalues=d[["pvalue"]].values,
    column_labels=[""],
    title="Dopaminergic synapse enrichment\n(DA-FASS vs bulk synaptosomes)",
    cbar_label="Log2 (DA-FASS / SYN)",
    filename="panel_b_pagetblanc",
)


In [ ]:
# Panel c — sorted ascending so the VMAT2 side (negative ratio) is at the top
d = asmerian.sort_values("LP2_effect")

heat_panel(
    proteins=d["protein"].tolist(),
    effects=d[["LP2_effect", "P4_effect"]].values,
    pvalues=d[["LP2_pvalue", "P4_pvalue"]].values,
    column_labels=["LP2\n(synaptic)", "P4\n(axonal)"],
    title="Vesicle-type partitioning\n(VMAT2+ vs VGLUT2+ SVs, striatum)",
    cbar_label="Log2 (VGLUT2 / VMAT2)",
    filename="panel_c_asmerian",
    figsize=(2.9, 4.4),
    arrows=[(0.68, 0.98, BLUE, "VMAT2 vesicle"),
            (0.32, 0.02, RED, "VGLUT2 vesicle")],
)
